<a href="https://colab.research.google.com/github/oooinr4018-web/-1/blob/main/ESAA%EA%B3%BC%EC%A0%9C_0911%EC%9D%98_%EC%82%AC%EB%B3%B8%EC%9D%98_%EC%82%AC%EB%B3%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 토픽 모델링 (Topic Modeling) - 뉴스그룹

토픽 모델링 (Topic Modeling): 문서 집합에 숨어 있는 주제 찾아내는 것

- 머신러닝 기반 토픽 모델링 적용

- 숨겨진 주제 표현 중심 단어 함축적 추줄

- 기법:  LSA(Latent Semantic nalysis), LDA(Latent Dirichlet Allocatin =/ Linear Discriminant Analysis)












# LDA

- LatentDirichletAllocation 클래스 제공

- Count 기반의 벡터화만 사용

(초기 버전 LDA의 토픽 모델링 제공 X -> gensim 패키지 등장과 함께 사이킷런의 LDA 제공)

- fetch_20newsgroups(): categories 파라미터로 필요한 주제 필터링 추출, Count 기반 벡터화 변환


토픽 모델링 예제) 20 뉴스그룹 데이터 세트

- 20가지 주제의 뉴스그룹 데이터

- 8개의 주제 추출 -> 텍스트에 LDA 기반 토픽 모델링 적용


In [ ]:
# max_features=1000으로 word 피처의 개수 제한
# ngram_range=(1,2)로 설정

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# 모터사이클, 야구, 그래픽스, 윈도우즈, 중동, 기독교. 전자공학. 의학 8개 주제를 추출.
cats=['rec.motorcycles', 'rec.sport.baseball', 'comp.graphics', 'comp.windows.x',
      'talk.politics.mideast', 'soc.religion.christian', 'sci.electronics', 'sci.med']

# 위에서 cats 변수로 기재된 카테고리만 추출. fetch_20newsgroups()의 categories에 cats 입력
news_df=fetch_20newsgroups(subset='all', remove=('headers', 'footers', 'quotes'),
                           categories=cats, random_state=0)
# LDA는 Count 기반의 벡터화만 적용합니다.
count_vect=CountVectorizer(max_df=0.95, max_features=1000, min_df=2, stop_words='english',
                           ngram_range=(1,2))
feat_vect=count_vect.fit_transform(news_df.data)
print('CountVectorizer Shape:', feat_vect.shape)

CountVectorizer Shape: (7862, 1000)


- CountVectorizer 객체 변수 feat_vct: 7862개의 문서, 1000개의 피처

In [ ]:
# 토픽 개수 8개
#n_components 파라미터로 토픽 개수조정
# 예제 수행할 때마다 결과 같게 하기 위해 같은 random_state

lda=LatentDirichletAllocation(n_components=8, random_state=0)
lda.fit(feat_vect)

LatentDirichletAllocation(n_components=8, random_state=0)

- LatentDirichletAllocation.fit(데이터 세트) 수행 -> LatentDirichletAoolcation 객체 components_ 속성값 가짐.

*components_: 개별 토픽별로 각 word 피처가 그 토픽에 얼만큼 할당되었는지에 대한 수치





In [ ]:
print(lda.components_.shape)
lda.components_

(8, 1000)


array([[2.69030238e+02, 1.87798026e+02, 7.09003824e+01, ...,
        1.22710343e+01, 1.06329639e+02, 7.25995512e+01],
       [1.25091799e-01, 2.46049106e+00, 1.25051902e-01, ...,
        2.80071176e+02, 1.25089783e-01, 5.05669662e+01],
       [1.33978420e+02, 1.25042012e-01, 9.98277256e+01, ...,
        1.25092219e-01, 3.31078261e+01, 1.25028398e-01],
       ...,
       [2.98813886e+01, 1.88071366e+01, 1.14748730e+01, ...,
        1.93022584e+01, 5.29368271e+00, 1.44478198e+01],
       [1.25074899e-01, 1.25105300e-01, 1.25004235e-01, ...,
        1.03576436e+02, 1.25100535e-01, 7.22276359e+01],
       [1.25172284e-01, 1.03967760e+00, 1.25221075e-01, ...,
        5.31740996e+01, 1.25025929e-01, 1.25062991e-01]])

- components_: array[8,1000]으로 구성

(8개의 토픽별로 1000개의 word 피처가 해당 토픽별로 연관도 값 가짐.)

- components_array_의 0번째 row, 10번째 col에 있는 값이, Topic #0에 대해 피처 벡터화된 행렬에서 10번째 칼럼에 해당하는 피처가 Topic#0에 연관되는 수치 값 가짐.

- lda_model.components_ 값만으로 각 토픽별 word 연관도 확인하기 어려움.

  



In [ ]:
# display_topics() 함수 생성하여, 각 토픽별 연관도가 높은 순으로 Word 나열

def display_topics(model, feature_names, no_top_words):
  for topic_index, topic in enumerate(model.components_):
    print('Topic #', topic_index)

    # components_array에서 가장 값이 큰 순으로 정렬했을 때, 그 값의 array 인덱스를 반환.
    topic_word_indexes=topic.argsort()[::-1]
    top_indexes=topic_word_indexes[:no_top_words]

    # top_indexes 대상인 인덱스별로 feature_names에 해당하는 word feature 추출 후 join으로 concat
    feature_concat=' '.join([feature_names[i] for i in top_indexes])
    print(feature_concat)

# CountVectorizer 객체 내의 전체 word이 명칭을 get_features_names()를 통해 추출
feature_names=count_vect.get_feature_names_out()

# 토픽별 가장 연관도가 높은 word를 15개만 추출
display_topics(lda, feature_names, 15)

Topic # 0
10 year medical health 1993 20 12 disease cancer team patients research number new 11
Topic # 1
don just like know think good time ve does way really people want ll right
Topic # 2
image file jpeg output program gif images format files color entry use bit 03 02
Topic # 3
armenian armenians turkish people said turkey armenia government genocide turks muslim russian greek azerbaijan killed
Topic # 4
israel jews dos jewish israeli dos dos arab state people arabs palestinian adl ed anti peace
Topic # 5
edu com available graphics ftp window use mail data motif software version pub information server
Topic # 6
god people jesus church believe say christ does christian think christians did know bible man
Topic # 7
thanks use using does help like display need problem know server screen windows window program


- Topic #0: 일부 불분명한 주제어, 주로 의학 관련 주제어

- Topic#1: 명확하지 않은 일반적인 단어

- Topic#2: 컴퓨터 그래픽스 영역의 주제어

- Topic#3: 일반적인 단어

- Topic#4: 중동 영역의 주제어

- Topic#5: 일부 컴퓨터 그래픽스 영역의 주제어, 전반적인 컴퓨터 관련 용어

- Topic#6: 기독교 관련 주제어

- Topic#7: 윈도우 운영체제와 관련된 주제어

- 전체적인 관점: Topic #1,3,5 애매한 주제어, 야구 주제의 경우 명확한 주제어 X



# 문서 유사도 측정 방법 -코사인 유사도

코사인 유사도(Cosine Similarity)를 이용한 문서 간 유사도 비교

- 코사인 유사도

: 벡터와 벡터 간 유사도 비교 시, 벡터의 크기보다 벡터의 상호 방향성이 얼마나 유사한지에 기반

: 희소 행렬 기반 문사와 문서 간 크기 기반 유사도 지표(유클리드거리 기반 지표)는 정확도가 낮음.

:두 벡터 사잇각 (유사 정도 수치로 표시)

- similarity = cos(세타) = 두 벡터의 내적을 총 벡터 크기의 합으로 나눈 값

- 상호 관계: 유사, 관련 X, 반대 관계




In [ ]:
# 서로 간의 문서 유사도를 코사인 유사도 기반으로 구해보기ㅣ

import numpy as np

def cos_similarity(v1, v2):
  dot_product=np.dot(v1,v2)
  l2_norm=(np.sqrt(sum(np.square(v1)))*np.sqrt(sum(np.square(v2))))
  similarity=dot_product/l2_norm

  return similarity
# doc_list로 정의된 3개의 간단한 문서의 유사도 비교
# 문서를 TF-IDF로 벡터화된 행렬로 변환

from sklearn.feature_extraction.text import TfidfVectorizer

doc_list=['if you take the blue pill, the story ends',
          'if you take the red pill, you stay in Wonderland',
          'if you take the red pill, I show you how deep the rabbit hole goes']

tfidf_vect_simple=TfidfVectorizer()
feature_vect_simple=tfidf_vect_simple.fit_transform(doc_list)
print(feature_vect_simple.shape)

(3, 18)


In [ ]:
# 반환 행렬은 희소 행렬
# cos_similarity() 함수의 인자인 array로 만들기 위해, 밀집 행렬로 변환 후 다시 각각을 배열로 변환

# TfidfVectorizer로 transform)한 결과는 희소 행렬이므로 밀집 행렬로 변환.
feature_vect_dense=feature_vect_simple.todense()

# feature_vect_dense[0]은 doc_list 첫 번째 문서의 피처 벡터화
# feature_vect_dense[1]은 doc_list 두 번째 문서의 피처 벡터화
# 첫 번째 문장과 두 번째 문장의 피처 벡터 추출
vect1=np.array(feature_vect_dense[0]).reshape(-1, )
vect2=np.array(feature_vect_dense[1]).reshape(-1, )

# 첫 번째 문장과 두 번째 문장의 피처 벡터로 두 개 문장의 코사인 유사도 추출
similarity_simple=cos_similarity(vect1, vect2)
print('문장 1, 문장 2 cosine 유사도: {0:.3f}'.format(similarity_simple))


문장 1, 문장 2 cosine 유사도: 0.402


- 첫 번째 문장, 두 번째 문장의 코사인 유사도 = 0.402

In [ ]:
vect1=np.array(feature_vect_dense[0]).reshape(-1, )
vect3=np.array(feature_vect_dense[2]).reshape(-1, )
similarity_simple=cos_similarity(vect1, vect3)
print('문장 1, 문장 3 cosine 유사도: {0:.3f}'.format(similarity_simple))

vect2=np.array(feature_vect_dense[1]).reshape(-1, )
vect3=np.array(feature_vect_dense[2]).reshape(-1, )
similarity_simple=cos_similarity(vect2, vect3)
print('문장 2, 문장 3 cosine 유사도: {0:.3f}'.format(similarity_simple))

문장 1, 문장 3 cosine 유사도: 0.404
문장 2, 문장 3 cosine 유사도: 0.456


- 사이킷런의 코사인 유사도 측정: sklearn.metrics.pairwise.cosine_similarity API

- cosine_similarity()

(1) 희소 행렬, 밀집 행렬 모두 가능 & 행렬, 배열 모두 가능 (별도의 변환 과정 X)

(2) 함수의 두 개의 입력 파라미터

첫 번째 파라미터: 비교 기준이 되는 문서의 피처 행렬,

두 번째 파라미터: 비교되는 문서의 피처 행렬

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_simple_pair=cosine_similarity(feature_vect_simple[0], feature_vect_simple)
print(similarity_simple_pair)


[[1.         0.40207758 0.40425045]]


- 첫 번째 유사도 값 1: 비교 기준인 첫 번째 문서 자신에 대한 유사도 측정

- 두 번째 유사도 값 0.40207758: 첫 번째, 두 번째 문서의 유사도

- 세 번째 유사도 값0.40425045: 첫 번째, 세 번째 문서의 유사도



In [ ]:
# 유사도 측정 값 1을 제외

from sklearn.metrics.pairwise import cosine_similarity

similarity_simple_pair=cosine_similarity(feature_vect_simple[0], feature_vect_simple[1:])
print(similarity_simple_pair)

[[0.40207758 0.40425045]]


In [ ]:
# cosine_similarity()를 쌍으로 코사인 유사도 값 제공
# 1번째 문서와 2,3번째 문서의 코사인 유사도
# 2번째 문서와 1,3번째 문서의 코사인 유사도
# 3번째 문서와 1,2번째 문서의 코사인 유사도

similarity_simple_pair=cosine_similarity(feature_vect_simple, feature_vect_simple)
print(similarity_simple_pair)
print('shape:', similarity_simple_pair.shape)


[[1.         0.40207758 0.40425045]
 [0.40207758 1.         0.45647296]
 [0.40425045 0.45647296 1.        ]]
shape: (3, 3)


In [ ]:
- 반환 값 (3.3) 형태의 ndarray

- 첫 번째 row: 1번 문서와 2,3번째 문서의 코사인 유사도

- 두 번째 row: 2번 문서와 1,3번째 문서의 코사인 유사도

- 세 번째 row: 3번 문서와 1,2번째 문서의 코사인 유사도